# Fit timings

How long the volume and surface greedy fits take on the prepared models, for a few item budgets. 

In [1]:
from pathlib import Path
import sys
from time import perf_counter

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

root = next(candidate for parent in (Path.cwd(), *Path.cwd().parents)
            for candidate in (parent, parent / "mesh-to-housing")
            if (candidate / "src/mesh_to_housing").is_dir())
sys.path.insert(0, str(root / "src"))

from mesh_to_housing import (
    CombinedLossConfig, Cuboid, ExcessSurfaceConfig, Execution, FittingTask, Greedy, LossTerm, MeshLoader,
    OutsideVolumeConfig, RandomPool, ResidualProposals, TransformConstraints, UncoveredSurfaceConfig,
    UncoveredVolumeConfig, VoxelGridConfig, fit, resize_mesh,
)
from mesh_to_housing.geometry.voxel_grid import VoxelGrid

pio.renderers.default = "plotly_mimetype+notebook"
y_up_to_z_up = np.array([[1, 0, 0, 0], [0, 0, -1, 0], [0, 1, 0, 0], [0, 0, 0, 1]], dtype=float)

In [2]:
models = {
    "dinosaur": "z",
    "chair": "y",
    "raven": "y",
    "lion": "z",
    "biplane": "y",
}
model_size = 10.0
budgets = [50, 100, 200]
seed = 42
workers = 1

catalog = (Cuboid("floor_tile", dimensions=(2.0, 2.0, 0.43322)),)
constraints = TransformConstraints(scale_bounds=(0.2, 2.0))
grid = VoxelGridConfig(resolution=64, gap_closing=0.0)
pool = RandomPool(grid=grid, centers=256, orientations=8, scales=5)
fits = {
    "Volume": (Greedy(pool=pool), CombinedLossConfig(terms={
        "uncovered": LossTerm(UncoveredVolumeConfig(grid=grid)),
        "outside": LossTerm(OutsideVolumeConfig(grid=grid)),
    })),
    "Surface": (Greedy(pool=pool, renewed=ResidualProposals(grid=grid, skin_only=True, spots=32, spins=2, scales=5),
                       renew_every=10), CombinedLossConfig(terms={
        "uncovered": LossTerm(UncoveredSurfaceConfig(grid=grid, skin_only=True)),
        "excess": LossTerm(ExcessSurfaceConfig(grid=grid)),
    })),
}

In [3]:
rows = []
for model, up in models.items():
    mesh = MeshLoader.load(root / "mesh-assets/prepared" / f"{model}.ply")
    if up == "y":
        mesh.apply_transform(y_up_to_z_up)
    target = resize_mesh(mesh, model_size)
    started = perf_counter()
    VoxelGrid.from_mesh(target, grid)
    grid_seconds = perf_counter() - started
    for budget in budgets:
        task = FittingTask(target, catalog, budget, constraints)
        for name, (algorithm, objective) in fits.items():
            started = perf_counter()
            with Execution(workers=workers):
                construction = fit(task, algorithm=algorithm, loss=objective.build(), seed=seed)
            seconds = perf_counter() - started
            rows.append({"Model": model, "Budget": budget, "Fit": name, "Items": len(construction),
                         "Fit (s)": seconds, "Grid (s)": grid_seconds})
            print(f"{model} · {budget} items · {name}: {seconds:.1f} s")
timings = pd.DataFrame(rows)

dinosaur · 50 items · Volume: 2.7 s
dinosaur · 50 items · Surface: 4.8 s
dinosaur · 100 items · Volume: 2.2 s
dinosaur · 100 items · Surface: 5.2 s
dinosaur · 200 items · Volume: 2.3 s
dinosaur · 200 items · Surface: 7.0 s
chair · 50 items · Volume: 3.0 s
chair · 50 items · Surface: 4.4 s
chair · 100 items · Volume: 3.1 s
chair · 100 items · Surface: 5.3 s
chair · 200 items · Volume: 3.1 s
chair · 200 items · Surface: 6.9 s
raven · 50 items · Volume: 2.9 s
raven · 50 items · Surface: 4.4 s
raven · 100 items · Volume: 3.0 s
raven · 100 items · Surface: 5.2 s
raven · 200 items · Volume: 3.1 s
raven · 200 items · Surface: 6.9 s
lion · 50 items · Volume: 4.6 s
lion · 50 items · Surface: 4.4 s
lion · 100 items · Volume: 4.6 s
lion · 100 items · Surface: 5.4 s
lion · 200 items · Volume: 4.8 s
lion · 200 items · Surface: 6.8 s
biplane · 50 items · Volume: 7.8 s
biplane · 50 items · Surface: 4.5 s
biplane · 100 items · Volume: 8.3 s
biplane · 100 items · Surface: 5.2 s
biplane · 200 items · Vo